In [ ]:
# Импортируем стандартные библиотеки для анализа данных и визуализации
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Загружаем датасет Telco Churn по прямой ссылке
url = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'
df = pd.read_csv(url)

# Приводим TotalCharges к числовому типу (ошибки заменяем нулями)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)

# Бинаризуем целевую переменную Churn (переводим 'Yes'/'No' в 1/0) с помощью map
df['Churn_bin'] = df['Churn'].map({'No': 0, 'Yes': 1})


In [ ]:
# Выделяем числовые признаки в отдельный датафрейм num
num = df[['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen']]

# Кодируем категориальные признаки с помощью One-Hot Encoding, удаляя первые столбцы
cat = pd.get_dummies(df[['Contract', 'PaymentMethod']], drop_first=True)

# Объединяем числовые и закодированные признаки по столбцам (axis=1) в матрицу X
X = pd.concat([num, cat], axis=1)

# Выносим бинаризованный целевой признак в вектор y
y = df['Churn_bin']

# Выводим итоговую размерность получившегося набора признаков X
print('X shape:', X.shape)

# Отображаем полный список названий колонок (фичей), вошедших в финальный набор
print('Features:', list(X.columns))

In [ ]:
# Импортируем функцию оценки взаимной информации для классификации
from sklearn.feature_selection import mutual_info_classif

# Вычисляем MI-показатели для всех признаков матрицы X относительно целевой переменной y
mi = mutual_info_classif(X, y, random_state=42)

# Создаем объект Series, где индексами будут названия признаков, и сортируем их по убыванию
mi_scores = pd.Series(mi, index=X.columns).sort_values(ascending=False)

# Выводим показатели на экран, округляя их до четырех знаков после запятой
mi_scores.round(4)

In [ ]:
# Задаем фиксированный размер окна для графика (8 на 5 дюймов)
plt.figure(figsize=(8, 5))

# Строим горизонтальную столбчатую диаграмму (barh) приятного стального синего цвета
mi_scores.plot(kind='barh', color='steelblue')

# Настраиваем заголовки графика и подписи осей для наглядности
plt.title('Mutual information with Churn')
plt.xlabel('MI score')

# Корректируем автоматические отступы и выводим график на экран
plt.tight_layout()
plt.show()

In [ ]:
# Импортируем модули для отбора признаков, логистической регрессии и масштабирования
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# Масштабируем признаки матрицы X, так как логистическая регрессия чувствительна к шкале данных
X_scaled = StandardScaler().fit_transform(X)

# Инициализируем RFE с базовой моделью логистической регрессии и задаем отбор топ-3 признаков
rfe = RFE(LogisticRegression(max_iter=1000), n_features_to_select=3)
rfe.fit(X_scaled, y)

# Создаем результирующий DataFrame, показывающий статус отбора (kept) и ранг (rank) каждого признака
result = pd.DataFrame({'feature': X.columns, 'kept': rfe.support_, 'rank': rfe.ranking_})

# Сортируем таблицу по рангу, чтобы топ-3 выбранных признака (с rank=1) оказались вверху
result.sort_values('rank')

In [ ]:
# Импортируем функцию разделения выборки и метрику accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Разделяем масштабированные признаки на обучающую и тестовую выборки (80/20)
X_tr, X_te, y_tr, y_te = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Обучаем логистическую регрессию на ВСЕХ признаках и выводим точность на тестовой выборке
m_all = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
print('All features:', round(accuracy_score(y_te, m_all.predict(X_te)), 4))

# Находим индексы трех лучших признаков, отобранных алгоритмом RFE
idx = np.where(rfe.support_)[0]

# Обучаем модель ТОЛЬКО на этих трех лучших признаках и выводим её точность
m_top = LogisticRegression(max_iter=1000).fit(X_tr[:, idx], y_tr)
print('Top 3:', round(accuracy_score(y_te, m_top.predict(X_te[:, idx])), 4))